# Quick SLM — 06 · Re-grade Cached Outputs (single judge)

Re-scores **already-generated** probe outputs with one consistent judge, **`google/gemma-4-31B-it-qat-q4_0-unquantized`**. It does **not** load Quick SLM and does **not** regenerate any text — it only re-reads the cached `logs/probe_outputs_<label>.json` files that `03_checkpoint_test.ipynb` already wrote.

## Why this exists

The early checkpoints (through ~step 5000) were graded by a **different judge** than the later ones, so the comparison table in `03` mixed two scorers and the numbers were not comparable. This notebook re-grades **every** checkpoint with the same Gemma Q4 judge, overwriting `logs/scored_outputs_<label>.json` in place so the table is uniform.

## Pipeline

1. Find every `logs/probe_outputs_<label>.json` (the cached generations — untouched).
2. Load Gemma Q4 once.
3. Re-score every `(prompt, expected, response)` tuple 0-5. **Overwrite** each `logs/scored_outputs_<label>.json` in place, no regeneration.
4. Rebuild the comparison table → `logs/eval_table.{json,csv}`.

## Important

- **Overwrites in place.** Existing `scored_outputs_<label>.json` files are replaced. The old scores are not kept.
- The rubric, judge model, and `gemma_score` function are **byte-identical** to `03_checkpoint_test.ipynb`, so re-grading here and grading a fresh checkpoint in `03` produce comparable scores.
- No Quick SLM checkpoint is loaded; the probe outputs are read from disk exactly as generated.


## 1. Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies


In [ ]:
# Only the judge stack is needed here — no Quick SLM, no generation.
# bitsandbytes provides the 4-bit quantization for the Gemma judge; pandas builds
# the final table. torchvision/torchaudio are left as Colab ships them.
!pip install -q --upgrade transformers accelerate safetensors tqdm pandas bitsandbytes

## 3. Locate the cached probe outputs

Globs `logs/probe_outputs_*.json` — the generations `03` already produced. Nothing here regenerates them. For each, we also peek at any existing `scored_outputs_<label>.json` and report which judge last scored it, so the mismatch this notebook fixes is visible before we start.


In [ ]:
import os, json
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/quick-slm')
LOGS_DIR   = DRIVE_ROOT / 'logs'

# The single judge every checkpoint will be re-scored with.
GEMMA_MODEL = 'google/gemma-4-31B-it-qat-q4_0-unquantized'

if not LOGS_DIR.exists():
    raise FileNotFoundError(
        f'No logs dir at {LOGS_DIR}. Run 03_checkpoint_test.ipynb Phase 1 first '
        f'to generate probe_outputs_*.json.')

def label_of(probe_path):
    # probe_outputs_<label>.json  ->  <label>
    return probe_path.name[len('probe_outputs_'):-len('.json')]

def probe_outputs_path(label):
    return LOGS_DIR / f'probe_outputs_{label}.json'

def scored_path(label):
    return LOGS_DIR / f'scored_outputs_{label}.json'

probe_files = sorted(LOGS_DIR.glob('probe_outputs_*.json'))
if not probe_files:
    raise FileNotFoundError(
        f'No probe_outputs_*.json under {LOGS_DIR}. Run 03 Phase 1 first.')

# Load the cached generations and report the current (possibly inconsistent) judge.
ALL_OUTPUTS = []
print(f'found {len(probe_files)} cached checkpoints in {LOGS_DIR}:')
print(f'{"label":<22s} {"probes":>6s}  current judge')
print('-' * 78)
for pf in probe_files:
    rec = json.loads(pf.read_text())
    label = rec.get('label', label_of(pf))
    ALL_OUTPUTS.append(rec)
    sp = scored_path(label)
    if sp.exists():
        prev = json.loads(sp.read_text()).get('judge', '(unrecorded)')
    else:
        prev = '(never scored)'
    flag = '' if prev == GEMMA_MODEL else '  <- will change'
    print(f'{label:<22s} {len(rec.get("outputs", [])):>6d}  {prev}{flag}')

print('-' * 78)
print(f'\nall of the above will be re-scored with:\n  {GEMMA_MODEL}')

## 4. Load the Gemma judge

Identical load, rubric, and scoring function to `03_checkpoint_test.ipynb` Phase 2. Google's QAT Q4 checkpoint loaded in **4-bit** nf4 via bitsandbytes (~18 GB in VRAM). BF16 is banned for this model — the full weights (~62 GB) leave too little headroom on the RTX PRO 6000 once activations and the KV cache are added. Quantization-aware training means the Q4 weights were tuned for 4-bit, so judge quality is barely affected. Open access — no HuggingFace token needed.


In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device : {device}')
if device == 'cuda':
    print(f'gpu    : {torch.cuda.get_device_name(0)}')

# 4-bit nf4; device_map='auto' places the quantized weights on the GPU. Compute
# stays bf16. Don't pass a plain dtype alongside this.
_quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'loading judge: {GEMMA_MODEL} (4-bit nf4)')
gemma_tok = AutoTokenizer.from_pretrained(GEMMA_MODEL)
gemma     = AutoModelForCausalLM.from_pretrained(
    GEMMA_MODEL, quantization_config=_quant_cfg, device_map='auto'
)
gemma.eval()
total_b = sum(p.numel() for p in gemma.parameters()) / 1e9
print(f'  loaded — {total_b:.1f} B params (dense)')

# Rubric is byte-identical to 03 so scores here match scores there.
JUDGE_RUBRIC = """You are evaluating outputs from a small (103M parameter) language model that's still in pretraining. The model has not been fine-tuned, so most outputs will be poor.

Score the response from 0 to 5:
- 0: nonsensical, empty, or pure repetition loop
- 1: grammatical English but topically unrelated to the prompt
- 2: on-topic but factually or structurally wrong
- 3: partially correct or relevant; missing or wrong in specific places
- 4: mostly correct with minor flaws
- 5: fully correct and well-formed

Be strict. Most outputs from an early-training model should score 0-2.

PROMPT:
{prompt}

EXPECTED (what a good answer looks like):
{expected}

MODEL RESPONSE:
{response}

Respond with EXACTLY this format and nothing else:
SCORE: <integer 0-5>
REASON: <one short sentence>"""

_SCORE_RE  = re.compile(r'SCORE:\s*(\d+)', re.IGNORECASE)
_REASON_RE = re.compile(r'REASON:\s*(.+)', re.IGNORECASE)

@torch.no_grad()
def gemma_score(prompt, expected, response):
    """Run Gemma judge on one (prompt, expected, response). Returns (score:int, reason:str).

    score = -1 if Gemma's output didn't parse.
    Truncates response to 800 chars to keep judge prompts bounded.
    """
    msg = JUDGE_RUBRIC.format(prompt=prompt[:800], expected=expected, response=response[:800])
    chat = [{'role': 'user', 'content': msg}]

    # Two-step: chat template → text → tokenize. Robust across transformers
    # versions (newer ones return BatchEncoding from apply_chat_template,
    # older ones return raw tensors — splitting avoids the .shape pitfall).
    text = gemma_tok.apply_chat_template(
        chat, tokenize=False, add_generation_prompt=True
    )
    enc = gemma_tok(text, return_tensors='pt').to(gemma.device)
    prompt_len = enc['input_ids'].shape[1]

    out = gemma.generate(
        **enc,
        max_new_tokens=120, do_sample=False,
        pad_token_id=gemma_tok.pad_token_id or gemma_tok.eos_token_id,
    )
    decoded = gemma_tok.decode(out[0, prompt_len:], skip_special_tokens=True)
    score_m  = _SCORE_RE.search(decoded)
    reason_m = _REASON_RE.search(decoded)
    score    = int(score_m.group(1)) if score_m else -1
    reason   = reason_m.group(1).strip() if reason_m else decoded.strip()[:200]
    return max(-1, min(score, 5)), reason

# Quick smoke test on a known case
_s, _r = gemma_score('The capital of France is', 'Paris.', 'Paris.')
print(f'\nsmoke test  : score={_s}  reason={_r!r}')

## 5. Re-grade every cached checkpoint

For each `probe_outputs_<label>.json`, re-score all of its probes with Gemma Q4 and **overwrite** `scored_outputs_<label>.json` in place. Unlike `03`, this does **not** skip files that already exist — the whole point is to replace the stale scores. Set `ONLY_MISMATCHED = True` to re-grade only the checkpoints whose current judge differs from Gemma Q4 (faster, if you trust the files already tagged with this judge).


In [ ]:
from tqdm.auto import tqdm

# False -> re-grade everything (uniform, recommended for a clean table).
# True  -> skip files already scored by GEMMA_MODEL, re-grade only the rest.
ONLY_MISMATCHED = False

ALL_SCORED = []
for record in ALL_OUTPUTS:
    label = record.get('label', label_of(probe_outputs_path('')))
    out_path = scored_path(label)

    if ONLY_MISMATCHED and out_path.exists():
        prev = json.loads(out_path.read_text())
        if prev.get('judge') == GEMMA_MODEL:
            print(f'skip {label} — already scored by {GEMMA_MODEL}')
            ALL_SCORED.append(prev)
            continue

    print(f'\nre-grading {label} ({len(record["outputs"])} probes) -> overwriting {out_path.name}')
    scored = {'label': label, 'hf_dir': record.get('hf_dir'),
              'judge': GEMMA_MODEL, 'regraded': True, 'outputs': []}
    for o in tqdm(record['outputs'], desc=f'  {label}', leave=False):
        score, reason = gemma_score(o['prompt'], o['expected'], o['response'])
        scored['outputs'].append({**o, 'score': score, 'reason': reason})
    out_path.write_text(json.dumps(scored, indent=2))  # overwrite in place
    ALL_SCORED.append(scored)

print(f'\nre-graded {len(ALL_SCORED)} checkpoints with a single judge')
print(f'all scored_outputs_*.json in {LOGS_DIR} now carry judge = {GEMMA_MODEL}')

## 6. Comparison table

Aggregates the re-graded scores per checkpoint × category — every cell now comes from the same judge. Saved to `logs/eval_table.{json,csv}`, overwriting the mixed-judge table `03` left behind.


In [ ]:
import pandas as pd
from collections import defaultdict

CATEGORIES = ['fluency', 'knowledge', 'math', 'code', 'tool']

def step_of(label):
    # Mirror the step values 03's find_all_checkpoints() assigned, so the table
    # orders base -> base-final -> sft -> sft-final off the label alone.
    if label == 'final':
        return 10**9
    if label == 'sft_final':
        return 3 * 10**9
    if label.startswith('sft_step_'):
        return 2 * 10**9 + int(label.split('_')[-1])
    return int(label.split('_')[-1])

rows = []
for s in ALL_SCORED:
    by_cat = defaultdict(list)
    for o in s['outputs']:
        if o.get('score', -1) >= 0:
            by_cat[o['category']].append(o['score'])
    row = {'label': s['label'], 'step': step_of(s['label'])}
    all_scores = []
    for cat in CATEGORIES:
        if by_cat[cat]:
            row[cat] = round(sum(by_cat[cat]) / len(by_cat[cat]), 2)
            all_scores.extend(by_cat[cat])
        else:
            row[cat] = None
    row['overall'] = round(sum(all_scores) / max(len(all_scores), 1), 2) if all_scores else None
    rows.append(row)

df = pd.DataFrame(rows).sort_values('step').reset_index(drop=True)
display_cols = ['label', 'step'] + CATEGORIES + ['overall']
print('\nQuick SLM — capability emergence, uniformly re-graded (Gemma Q4, 0-5)')
print('=' * 96)
print(df[display_cols].to_string(index=False))
print('=' * 96)

(LOGS_DIR / 'eval_table.json').write_text(df.to_json(orient='records', indent=2))
df.to_csv(LOGS_DIR / 'eval_table.csv', index=False)
print(f'\nsaved: {LOGS_DIR}/eval_table.json')
print(f'saved: {LOGS_DIR}/eval_table.csv')